# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashritha-boop/fly_machine/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

For the Week-4 refresh lane, the Week-5 model keeps the baseline's three observable March inputs: `gsc_impressions_march`, `gsc_avg_position_march`, and `content_age_days`. The unit is one content item at the March snapshot. These are numeric features; no IDs, April outcomes, trend-derived fields, or product decisions enter the vector. Logistic Regression uses `log1p` impressions and imputes missing values inside a training-only pipeline.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from IPython.display import display

FEATURES = [
    "gsc_impressions_march",
    "gsc_avg_position_march",
    "content_age_days",
]
TARGET = "is_declining_label"
GROUP_ONLY = ["client_hash_id", "content_hash_id"]

feature_vector_spec = pd.DataFrame([
    {"feature": "gsc_impressions_march", "type": "numeric", "engineering": "log1p for Logistic Regression", "prediction-time window": "2026-03"},
    {"feature": "gsc_avg_position_march", "type": "numeric", "engineering": "0 treated as missing; median imputation fitted on training clients", "prediction-time window": "2026-03"},
    {"feature": "content_age_days", "type": "numeric", "engineering": "used as observed; median imputation fitted on training clients if needed", "prediction-time window": "as of 2026-03-31"},
])
print("One row = one content item; model inputs:")
display(feature_vector_spec)
print(f"Feature count: {len(FEATURES)}; target: {TARGET}; group-only fields: {GROUP_ONLY}")

One row = one content item; model inputs:


,feature,type,engineering,prediction-time window
0,gsc_impressions_march,numeric,log1p for Logistic Regression,2026-03
1,gsc_avg_position_march,numeric,0 treated as missing; median imputation fitted...,2026-03
2,content_age_days,numeric,used as observed; median imputation fitted on ...,as of 2026-03-31


Feature count: 3; target: is_declining_label; group-only fields: ['client_hash_id', 'content_hash_id']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

- `gsc_impressions_march`: March search exposure; the Week-4 extract keeps rows with at least 10 impressions, so it is observed before the April outcome. Model transform: `log1p` to reduce the heavy tail.
- `gsc_avg_position_march`: March average search position; zero means no position data and is converted to missing. The model imputes the median learned from training clients only.
- `content_age_days`: age as of March 31, derived from content creation date; known by the decision point. Missing values, if any, receive the training-only median.

All three are numeric, so no categorical encoding is needed. The Week-5 code fits imputation and scaling only on training clients; the test clients do not influence preprocessing.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_notes = pd.DataFrame([
    {"feature": "gsc_impressions_march", "meaning": "Search impressions in March", "missing / special value handling": "Rows are filtered to >=10; model applies log1p", "available before April label": True},
    {"feature": "gsc_avg_position_march", "meaning": "Mean search position in March", "missing / special value handling": "0 becomes missing; training median imputation", "available before April label": True},
    {"feature": "content_age_days", "meaning": "Age as of 2026-03-31", "missing / special value handling": "Training median imputation if missing", "available before April label": True},
])
assert feature_notes["available before April label"].all()
assert feature_notes["feature"].tolist() == FEATURES
print("Feature timing and treatment notes:")
display(feature_notes)
print("Categorical fields used: none")

Feature timing and treatment notes:


,feature,meaning,missing / special value handling,available before April label
0,gsc_impressions_march,Search impressions in March,Rows are filtered to >=10; model applies log1p,True
1,gsc_avg_position_march,Mean search position in March,0 becomes missing; training median imputation,True
2,content_age_days,Age as of 2026-03-31,Training median imputation if missing,True


Categorical fields used: none


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

The decision point is after March 31; the outcome is April impressions. I attack the declared model inputs for target fields, future-window fields, and identifiers. The assertions below are a feature-contract check; the Week-5 extract separately constructs the target from April and sends only `FEATURES` to the model.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
FEATURE_WINDOW = "2026-03"
LABEL_WINDOW = "2026-04"
TARGET_DEFINITION = "April impressions < 0.80 * March impressions"

FORBIDDEN_FEATURES = {
    "is_declining_label",
    "gsc_impressions_april",
    "trend_direction",
    "trend_pct",
    "client_hash_id",
    "content_hash_id",
}
assert set(FEATURES).isdisjoint(FORBIDDEN_FEATURES)
assert LABEL_WINDOW > FEATURE_WINDOW
assert TARGET not in FEATURES
assert "April" in TARGET_DEFINITION and "March" in TARGET_DEFINITION
assert "gsc_impressions_april" not in FEATURES

print("Leakage-contract checks passed:")
print(f"  Features are limited to {FEATURE_WINDOW} / pre-decision content age.")
print(f"  Label is defined only from {LABEL_WINDOW}: {TARGET_DEFINITION}.")
print("  IDs are reserved for grouping; trend-derived and future fields are excluded.")
print("  This is a schema/timing check, not a causal or empirical leakage experiment.")

Leakage-contract checks passed:
  Features are limited to 2026-03 / pre-decision content age.
  Label is defined only from 2026-04: April impressions < 0.80 * March impressions.
  IDs are reserved for grouping; trend-derived and future fields are excluded.
  This is a schema/timing check, not a causal or empirical leakage experiment.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- `client_hash_id`, `content_hash_id`: pseudonymous identifiers, used only to group clients and identify the row grain.
- `gsc_impressions_april`, `is_declining_label`: future outcome and its direct label; never available as prediction-time features.
- `trend_direction`, `trend_pct`: derived from recent impression movement and explicitly disallowed because they can encode the target.
- `baseline_score`, action/reason flags: outputs of the Week-4 rule, not independent evidence; including them would teach the model to copy the rule.
- March clicks and sessions: observable, but omitted so the learned model uses the same core evidence available to the Week-4 baseline rather than gaining a feature advantage.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields = pd.DataFrame([
    {"field": "client_hash_id / content_hash_id", "reason": "Group/split and row identity only; codes are not predictive features."},
    {"field": "gsc_impressions_april / is_declining_label", "reason": "April outcome and target itself; direct leakage."},
    {"field": "trend_direction / trend_pct", "reason": "Derived from trend windows and can reveal the target."},
    {"field": "baseline_score / action flags", "reason": "Rule outputs; using them would reproduce the baseline rather than learn from observations."},
    {"field": "gsc_clicks_march / ga4_sessions_march", "reason": "Safe March context, but omitted to keep the model input set aligned with the Week-4 baseline."},
])
assert set(FEATURES).isdisjoint({
    "client_hash_id", "content_hash_id", "gsc_impressions_april",
    "is_declining_label", "trend_direction", "trend_pct", "baseline_score",
    "action_label", "reason_code", "gsc_clicks_march", "ga4_sessions_march",
})
print("Excluded fields and reasons:")
display(excluded_fields)


Excluded fields and reasons:


,field,reason
0,client_hash_id / content_hash_id,Group/split and row identity only; codes are n...
1,gsc_impressions_april / is_declining_label,April outcome and target itself; direct leakage.
2,trend_direction / trend_pct,Derived from trend windows and can reveal the ...
3,baseline_score / action flags,Rule outputs; using them would reproduce the b...
4,gsc_clicks_march / ga4_sessions_march,"Safe March context, but omitted to keep the mo..."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.